In [ ]:
import os
import torch
import subprocess

# Check GPU
print("="*50)
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    total_mem = torch.cuda.get_device_properties(0).total_memory
    print(f"VRAM per GPU: {total_mem / 1e9:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

# Check disk space
result = subprocess.run(['df', '-h', '/teamspace/studios/this_studio'], 
                       capture_output=True, text=True)
print("\nDisk Space:")
print(result.stdout)

# Check RAM
result = subprocess.run(['free', '-h'], capture_output=True, text=True)
print("RAM:")
print(result.stdout)

In [ ]:
# Install all required packages
# lightning.ai already has torch, torchvision, numpy pre-installed

!pip install -q \
    omegaconf \
    h5py \
    hdf5plugin \
    zstandard \
    huggingface_hub \
    tqdm \
    scikit-learn \
    imageio \
    wandb

print("All packages installed!")

# Verify key imports
import torch
import h5py
import omegaconf
import huggingface_hub
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
import os
import sys

# Clone or update repo
!git clone https://github.com/krishbansal-2205/lewm-pushT.git /teamspace/studios/this_studio/lewm-pushT || echo "Repo already exists, pulling latest..."
%cd /teamspace/studios/this_studio/lewm-pushT
!git pull origin main || echo "Pull skipped"
sys.path.insert(0, '/teamspace/studios/this_studio/lewm-pushT')

# Verify structure
for root, dirs, files in os.walk('/teamspace/studios/this_studio/lewm-pushT'):
    # Skip hidden folders and caches
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__' and d != 'venv']
    level = root.replace('/teamspace/studios/this_studio/lewm-pushT', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for file in files:
            print(f"{subindent}{file}")

In [ ]:
import os

# Set data directory
os.environ["LEWM_DATA_DIR"] = "/teamspace/studios/this_studio/lewm-pushT/dataset"
os.makedirs("/teamspace/studios/this_studio/lewm-pushT/dataset", exist_ok=True)

# Check if dataset already cached (useful if running multiple times)
h5_path = "/teamspace/studios/this_studio/lewm-pushT/dataset/pusht_expert_train.h5"

if os.path.exists(h5_path):
    size_gb = os.path.getsize(h5_path) / 1e9
    print(f"Dataset already exists: {size_gb:.2f} GB")
else:
    print("Downloading dataset from HuggingFace...")
    print("This will take 10-20 minutes (12GB compressed)")
    
    !python -m data.download --data-dir /teamspace/studios/this_studio/lewm-pushT/dataset
    
    if os.path.exists(h5_path):
        size_gb = os.path.getsize(h5_path) / 1e9
        print(f"Download complete: {size_gb:.2f} GB")
    else:
        print("ERROR: Download failed!")

In [ ]:
import h5py
import numpy as np

h5_path = "/teamspace/studios/this_studio/lewm-pushT/dataset/pusht_expert_train.h5"

with h5py.File(h5_path, 'r') as f:
    print("Dataset structure:")
    print("=" * 40)
    for key in f.keys():
        dataset = f[key]
        print(f"  {key}:")
        print(f"    Shape: {dataset.shape}")
        print(f"    Dtype: {dataset.dtype}")
        if hasattr(dataset, 'nbytes'):
            print(f"    Size:  {dataset.nbytes / 1e9:.2f} GB")
    
    # FIX: Use correct key names ('pixels' not 'observations', 'action' not 'actions')
    obs_key = 'pixels' if 'pixels' in f else 'observations'
    act_key = 'action' if 'action' in f else 'actions'
    
    print(f"\nSample observation shape:", f[obs_key][0].shape)
    print(f"Observation dtype:", f[obs_key].dtype)
    print(f"Observation value range: [{f[obs_key][0].min()}, {f[obs_key][0].max()}]")
    print(f"Sample action:", f[act_key][0])
    print(f"Total samples:", f[obs_key].shape[0])
    
    if 'ep_offset' in f and 'ep_len' in f:
        print(f"\nEpisodes: {len(f['ep_offset'])}")
        print(f"Episode lengths: min={f['ep_len'][:].min()}, max={f['ep_len'][:].max()}, mean={f['ep_len'][:].mean():.1f}")

In [ ]:
# Write optimized config for lightning.ai H100
config_content = """
# LeWM Lightning AI H100 Configuration

# Data
data_dir: /teamspace/studios/this_studio/lewm-pushT/dataset
image_size: 224
train_split: 0.9
augmentation: true
num_workers: 8
cache_in_ram: true

# Model
latent_dim: 192
encoder_channels: [32, 64, 128, 256]
predictor_hidden: [512, 512, 512]
dropout: 0.1

# Training - optimized for H100 80GB VRAM
batch_size: 2048
epochs: 100
lr: 3e-4
weight_decay: 1e-4
grad_clip: 1.0
lambda_reg: 0.5
early_stopping_patience: 20
checkpoint_dir: /teamspace/studios/this_studio/lewm-pushT/checkpoints
use_amp: true

# SIGReg
sigreg_num_projections: 64

# Planner
cem_n_samples: 512
cem_top_k: 64
cem_n_iters: 5
cem_horizon: 10
action_dim: 2
action_low: -1.0
action_high: 1.0

# Evaluation
n_eval_episodes: 100
max_steps_per_episode: 50
success_threshold: 0.15

# Misc
seed: 42
device: cuda
log_every: 25
"""

os.makedirs('/teamspace/studios/this_studio/lewm-pushT/configs', exist_ok=True)
with open('/teamspace/studios/this_studio/lewm-pushT/configs/kaggle_t4.yaml', 'w') as f:
    f.write(config_content)

print("Config saved!")
print(config_content)

In [ ]:
# Quick sanity check - make sure everything works
import sys
sys.path.insert(0, '/teamspace/studios/this_studio/lewm-pushT')

import torch
from omegaconf import OmegaConf
from models.lewm import LeWM
from training.dataset import get_dataloaders

# Load config
config = OmegaConf.load('/teamspace/studios/this_studio/lewm-pushT/configs/kaggle_t4.yaml')
device = torch.device('cuda')

# Build model
print("Building model...")
model = LeWM(config).to(device)
params = model.count_parameters()
print(f"Parameters: {params['total']:,}")
print(f"  Encoder:   {params['encoder']:,}")
print(f"  Predictor: {params['predictor']:,}")

# Test forward pass with dummy data
print("\nTesting forward pass...")
obs = torch.randn(4, 3, 224, 224).to(device)
action = torch.randn(4, 2).to(device)
next_obs = torch.randn(4, 3, 224, 224).to(device)

with torch.no_grad():
    total_loss, pred_loss, reg_loss = model.compute_loss(obs, action, next_obs)

print(f"Total loss:  {total_loss.item():.4f}")
print(f"Pred loss:   {pred_loss.item():.4f}")
print(f"Reg loss:    {reg_loss.item():.4f}")

# Test AMP forward pass
print("\nTesting AMP (BF16) forward pass...")
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    total_loss, pred_loss, reg_loss = model.compute_loss(obs, action, next_obs)
print(f"AMP Total loss:  {total_loss.item():.4f}")
print(f"AMP OK!")

print("\nForward pass OK!")

# Test memory usage
if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1e9
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nVRAM used: {mem_used:.2f} / {mem_total:.1f} GB")

In [ ]:
from training.dataset import get_dataloaders
import time

print("Loading dataset (with RAM caching — this takes 2-5 minutes the first time)...")
t = time.time()

train_loader, val_loader, dataset = get_dataloaders(
    h5_path="/teamspace/studios/this_studio/lewm-pushT/dataset/pusht_expert_train.h5",
    batch_size=2048,
    train_split=0.9,
    augmentation=True,
    num_workers=8,
    image_size=224,
    seed=42,
    cache_in_ram=True,
)

print(f"Dataset loaded in {time.time()-t:.1f}s")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

# Test batch load speed (should be nearly instant with RAM cache)
t = time.time()
batch = next(iter(train_loader))
print(f"\nFirst batch loaded in {time.time()-t:.2f}s")
print(f"obs shape:      {batch['obs'].shape}")
print(f"action shape:   {batch['action'].shape}")
print(f"next_obs shape: {batch['next_obs'].shape}")
print(f"obs value range: [{batch['obs'].min():.2f}, {batch['obs'].max():.2f}]")
print(f"obs_raw value range: [{batch['obs_raw'].min():.2f}, {batch['obs_raw'].max():.2f}]")

In [ ]:
import os
import sys
sys.path.insert(0, '/teamspace/studios/this_studio/lewm-pushT')
os.chdir('/teamspace/studios/this_studio/lewm-pushT')

from omegaconf import OmegaConf
from models.lewm import LeWM
from training.train import set_seed, train_lewm
from training.dataset import get_dataloaders
import torch

# Load config
config = OmegaConf.load('configs/kaggle_t4.yaml')
device = torch.device('cuda')

# Set seed
set_seed(config.seed)

# Create checkpoint dir
os.makedirs(config.checkpoint_dir, exist_ok=True)

# Load data (RAM-cached — takes ~2-5 min the first time, instant after)
print("Loading data...")
train_loader, val_loader, dataset = get_dataloaders(
    h5_path=config.data_dir + "/pusht_expert_train.h5",
    batch_size=config.batch_size,
    train_split=config.train_split,
    augmentation=config.augmentation,
    num_workers=config.num_workers,
    image_size=config.image_size,
    seed=config.seed,
    cache_in_ram=getattr(config, 'cache_in_ram', True),
)

# Build model
print("Building model...")
model = LeWM(config).to(device)
print(f"Parameters: {model.count_parameters()['total']:,}")

# Train
print("\nStarting training...")
history = train_lewm(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    checkpoint_dir=config.checkpoint_dir,
)

print("\nTraining complete!")

In [ ]:
import torch
import matplotlib.pyplot as plt
import os

# Load history
history_path = "/teamspace/studios/this_studio/lewm-pushT/checkpoints/training_history.pt"
history = torch.load(history_path, weights_only=False)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs = range(1, len(history['train_loss']) + 1)

# Total loss
axes[0,0].plot(epochs, history['train_loss'], label='Train', color='blue')
axes[0,0].plot(epochs, history['val_loss'], label='Val', color='red')
axes[0,0].set_title('Total Loss')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Prediction loss
axes[0,1].plot(epochs, history['train_pred_loss'], label='Train', color='green')
axes[0,1].plot(epochs, history['val_pred_loss'], label='Val', color='orange')
axes[0,1].set_title('Prediction Loss (MSE)')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# SIGReg loss
axes[1,0].plot(epochs, history['train_reg_loss'], label='Train', color='purple')
axes[1,0].set_title('SIGReg Loss')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Latent std
axes[1,1].plot(epochs, history['latent_std'], color='teal')
axes[1,1].axhline(y=0.01, color='red', linestyle='--', label='Collapse threshold')
axes[1,1].set_title('Latent Std (Collapse Monitor)')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('LeWM Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/teamspace/studios/this_studio/lewm-pushT/checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved training_curves.png")